In [0]:
# O modelo dimensional parte da Gold já agregada e validada.

nome_tabela_gold = "workspace.gold.resumo_precos_semanais"

df_gold = spark.table(nome_tabela_gold)

print("Registro recebido da Gold:", df_gold.count())
print("Colunas recebidas:", len(df_gold.columns))

In [0]:
# Confiro a estrutura recebida da Gold antes de iniciar a criação das dimensões e da tabela fato.

df_gold.printSchema()

In [0]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Crio a dimensão de tempo com uma linha por combinação de ano e semana ISO.
# A chave substituta facilita o relacionamento com a futura tabela fato.

janela_tempo = Window.orderBy(
    "ano",
    "semana_iso"
)

dim_tempo = (
    df_gold
    .select(
        "ano",
        "semana_iso"
    )
    .distinct()
    .orderBy(
        "ano",
        "semana_iso"
    )
    .withColumn(
        "id_tempo",
        row_number().over(janela_tempo)
    )
    .select(
        "id_tempo",
        "ano",
        "semana_iso"
    )
)

print("Registros dim_tempo:", dim_tempo.count())

In [0]:
display(dim_tempo)

In [0]:
# Crio a dimensão de localidade com uma linha por município e estado.
# A chave substituta será usada depois na tabela fato.

janela_localidade = Window.orderBy(
    "estado",
    "municipio"
)

dim_localidade = (
    df_gold
    .select(
        "estado",
        "municipio"
    )
    .distinct()
    .orderBy(
        "estado",
        "municipio"
    )
    .withColumn(
        "id_localidade",
        row_number().over(janela_localidade)
    )
    .select(
        "id_localidade",
        "estado",
        "municipio"
    )
)

print("Registros dim_localidade:", dim_localidade.count())

In [0]:
display(dim_localidade.limit(20))

In [0]:
# Crio a dimensão de produto mantendo a unidade de medida junto do produto.
# Isso evita misturar produtos com unidades diferentes dentro do modelo.

janela_produto = Window.orderBy(
    "produto",
    "unidade_medida"
)

dim_produto = (
    df_gold
    .select(
        "produto",
        "unidade_medida"
    )
    .distinct()
    .orderBy(
        "produto",
        "unidade_medida"
    )
    .withColumn(
        "id_produto",
        row_number().over(janela_produto)
    )
    .select(
        "id_produto",
        "produto",
        "unidade_medida"
    )
)

print("Registros dim_produto:", dim_produto.count())

In [0]:
display(dim_produto)

In [0]:
# Crio a dimensão de bandeira com uma linha por bandeira comercial.
# Depois uso essa chave na tabela fato.

janela_bandeira = Window.orderBy(
    "bandeira"
)

dim_bandeira = (
    df_gold
    .select(
        "bandeira"
    )
    .distinct()
    .orderBy(
        "bandeira"
    )
    .withColumn(
        "id_bandeira",
        row_number().over(janela_bandeira)
    )
    .select(
        "id_bandeira",
        "bandeira"
    )
)

print("Registros dim_bandeira:", dim_bandeira.count())

In [0]:
display(dim_bandeira.limit(20))

In [0]:
# Relaciono a Gold com cada dimensão para substituir os atributos descritivos pelas respectivas chaves substitutas.

fato_preco_semanal = (
    df_gold

    .join(
        dim_tempo,
        on=[
            "ano",
            "semana_iso"
        ],
        how="left"
    )

    .join(
        dim_localidade,
        on=[
            "estado",
            "municipio"
        ],
        how="left"
    )

    .join(
        dim_produto,
        on=[
            "produto",
            "unidade_medida"
        ],
        how="left"
    )

    .join(
        dim_bandeira,
        on="bandeira",
        how="left"
    )

    .select(
        "id_tempo",
        "id_localidade",
        "id_produto",
        "id_bandeira",
        "quantidade_coletas",
        "preco_medio",
        "preco_minimo",
        "preco_maximo",
        "preco_mediano",
        "amplitude_preco"
    )
)

print(
    "Registros fato_preco_semanal:",
    fato_preco_semanal.count()
)

In [0]:
from pyspark.sql.functions import sum as spark_sum, when, col

# Confirmo que todos os registros da fato encontraram correspondência nas quatro dimensões.

display(
    fato_preco_semanal.select(
        spark_sum(
            when(col("id_tempo").isNull(), 1).otherwise(0)
        ).alias("id_tempo_nulos"),

        spark_sum(
            when(col("id_localidade").isNull(), 1).otherwise(0)
        ).alias("id_localidade_nulos"),

        spark_sum(
            when(col("id_produto").isNull(), 1).otherwise(0)
        ).alias("id_produto_nulos"),

        spark_sum(
            when(col("id_bandeira").isNull(), 1).otherwise(0)
        ).alias("id_bandeira_nulos")
    )
)

In [0]:
# Confirmo que a tabela fato continua representando todas as 19.515 coletas aprovadas da Silver.

display(
    fato_preco_semanal.agg(
        spark_sum("quantidade_coletas").alias("total_coletas")
    )
)

In [0]:
# Gravo a dimensão de tempo como tabela Delta gerenciada dentro da camada Gold do Unity Catalog.

nome_dim_tempo = "workspace.gold.dim_tempo"

(
    dim_tempo
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_dim_tempo)
)

print("dim_tempo gravada com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.gold.dim_tempo
        ORDER BY id_tempo
    """)
)

In [0]:
# Gravo a dimensão de localidade como tabela Delta gerenciada.

nome_dim_localidade = "workspace.gold.dim_localidade"

(
    dim_localidade
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_dim_localidade)
)

print("dim_localidade gravada com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            COUNT(DISTINCT id_localidade) AS ids_distintos
        FROM workspace.gold.dim_localidade
    """)
)

In [0]:
# Gravo a dimensão de produto como tabela Delta gerenciada.

nome_dim_produto = "workspace.gold.dim_produto"

(
    dim_produto
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_dim_produto)
)

print("dim_produto gravada com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            COUNT(DISTINCT id_produto) AS ids_distintos
        FROM workspace.gold.dim_produto
    """)
)

In [0]:
# Gravo a dimensão de bandeira como tabela Delta gerenciada.

nome_dim_bandeira = "workspace.gold.dim_bandeira"

(
    dim_bandeira
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_dim_bandeira)
)

print("dim_bandeira gravada com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            COUNT(DISTINCT id_bandeira) AS ids_distintos
        FROM workspace.gold.dim_bandeira
    """)
)

In [0]:
# Gravo a tabela fato como Delta gerenciada no Unity Catalog.

nome_fato_preco_semanal = "workspace.gold.fato_preco_semanal"

(
    fato_preco_semanal
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(nome_fato_preco_semanal)
)

print("fato_preco_semanal gravada com sucesso.")

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            SUM(quantidade_coletas) AS total_coletas,
            COUNT(DISTINCT id_tempo) AS tempos,
            COUNT(DISTINCT id_localidade) AS localidades,
            COUNT(DISTINCT id_produto) AS produtos,
            COUNT(DISTINCT id_bandeira) AS bandeiras
        FROM workspace.gold.fato_preco_semanal
    """)
)

In [0]:
# Valido se todas as chaves da fato realmente existem nas respectivas dimensões persistidas no Unity Catalog.

validacao_referencias = spark.sql("""
    SELECT
        SUM(CASE WHEN t.id_tempo IS NULL THEN 1 ELSE 0 END) AS tempo_sem_correspondencia,
        SUM(CASE WHEN l.id_localidade IS NULL THEN 1 ELSE 0 END) AS localidade_sem_correspondencia,
        SUM(CASE WHEN p.id_produto IS NULL THEN 1 ELSE 0 END) AS produto_sem_correspondencia,
        SUM(CASE WHEN b.id_bandeira IS NULL THEN 1 ELSE 0 END) AS bandeira_sem_correspondencia
    FROM workspace.gold.fato_preco_semanal AS f

    LEFT JOIN workspace.gold.dim_tempo AS t
        ON f.id_tempo = t.id_tempo

    LEFT JOIN workspace.gold.dim_localidade AS l
        ON f.id_localidade = l.id_localidade

    LEFT JOIN workspace.gold.dim_produto AS p
        ON f.id_produto = p.id_produto

    LEFT JOIN workspace.gold.dim_bandeira AS b
        ON f.id_bandeira = b.id_bandeira
""")

display(validacao_referencias)